# SteamMatch: Yorumlara Dayalı Oyun Öneri Sistemi
**Ders:** Veri Madenciliği (Data Mining) — Dönem Projesi  
**Problem Türü:** İşbirlikçi Filtreleme (Collaborative Filtering) Tabanlı Öneri Sistemi  
**Veri Seti:** Steam Review Dataset (Antoni Sobkowicz) — 21.7+ Milyon Satır  
**Kullanılan Kütüphane:** , , ,   

---

### Proje Kapsamı ve Hedefler
1. **Büyük Veri Yönetimi:** 21+ milyon satırlık ham veriden metin sütunlarını atlayarak düşük bellek tüketimiyle veri işleme.
2. **Duygu Durumu (Sentiment) Dönüşümü:** İkili duygu etiketlerinin (Recommended: True/False) 1-5 derecelendirme (rating) matrisine çevrilmesi.
3. **Seyreklik Azaltma (Sparsity Reduction):** En az 50 yoruma sahip oyunların ve en az 5 farklı oyunu oylamış kullanıcıların filtrelenmesi (1 Milyon+ satır garantisi ile).
4. **4 Farklı Algoritmanın Karşılaştırılması:**
   - **SVD (Singular Value Decomposition)**
   - **NMF (Non-negative Matrix Factorization)**
   - **KNNBasic (Cosine Benzerliği - Item-based)**
   - **NormalPredictor (Baseline - Rastgele Dağılım)**
5. **Model Değerlendirme:** 3-Katmanlı Çapraz Doğrulama (*3-Fold Cross-Validation*) ile **RMSE** ve **MAE** metrikleri üzerinden kıyaslama.


In [ ]:
# 1. Gerekli Kütüphanelerin Yüklenmesi
import os
import sys
import time
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Surprise kütüphanesi öneri algoritmaları ve veri yapıları
from surprise import SVD, NMF, KNNBasic, NormalPredictor
from surprise import Dataset, Reader
from surprise.model_selection import cross_validate

warnings.filterwarnings('ignore')
%matplotlib inline

print('✓ Tüm kütüphaneler başarıyla yüklendi.')


In [ ]:
# 2. Veri Setinin Yüklenmesi ve RAM Optimizasyonu
# Hocanın Kriteri: NLP/Metin sütunları (review) belleğe alınmayacak!
# usecols ile sadece kullanıcı ('author.steamid'), oyun ('app_id') ve duygu ('recommended') okunur.

DATA_PATH = 'data/steam_reviews.csv'
if not os.path.exists(DATA_PATH) and os.path.exists('data/dataset.csv'):
    DATA_PATH = 'data/dataset.csv'

print(f'Veri seti okunuyor: {DATA_PATH} ...')
start_time = time.time()

use_columns = ['author.steamid', 'app_id', 'recommended']
dtypes = {
    'author.steamid': 'int64',
    'app_id': 'int32',
    'recommended': 'bool'
}

df = pd.read_csv(DATA_PATH, usecols=use_columns, dtype=dtypes)
df = df.rename(columns={'author.steamid': 'user_id', 'app_id': 'game_id'})

load_time = time.time() - start_time
ram_usage = df.memory_usage(deep=True).sum() / (1024 * 1024)

print(f'✓ Toplam Satır Sayısı  : {len(df):,}')
print(f'✓ Yükleme Süresi        : {load_time:.2f} saniye')
print(f'✓ Bellek (RAM) Kullanımı: {ram_usage:.2f} MB')
df.head()


In [ ]:
# 3. Veri Ön İşleme ve Filtreleme (Sparsity Reduction)
# 3a. Sentiment -> Rating Dönüşümü: True (Pozitif) -> 5, False (Negatif) -> 1
df['rating'] = df['recommended'].map({True: 5, False: 1}).astype('int8')
df = df.drop(columns=['recommended'])

# Tekrarlanan (aynı kullanıcı, aynı oyun) kayıtlarını tekilleştir
df = df.drop_duplicates(subset=['user_id', 'game_id'], keep='last')

# 3b. Filtreleme 1: En az 50 yorum almış oyunlar
game_counts = df['game_id'].value_counts()
valid_games = game_counts[game_counts >= 50].index
df = df[df['game_id'].isin(valid_games)]

# 3c. Filtreleme 2: En az 5 farklı oyuna yorum yapmış kullanıcılar
user_counts = df.groupby('user_id')['game_id'].count()
valid_users = user_counts[user_counts >= 5].index
df = df[df['user_id'].isin(valid_users)]

# 3d. Zorunlu 1 Milyon Satır Kriteri Kontrolü
print(f'Filtreleme sonrası satır sayısı: {len(df):,}')
assert len(df) >= 1_000_000, 'HATA: Satır sayısı 1 milyonun altına düştü!'
print('✅ ZORUNLU KRİTER SAĞLANDI: Satır sayısı >= 1.000.000!')

# Modelleme için tam 1.000.000 satırlık deterministik örneklem
TARGET_ROWS = 1_000_000
df_model = df.sample(n=TARGET_ROWS, random_state=42).reset_index(drop=True)

print(f'
Modelleme Veri Kümesi Özeti:')
print(f'• Benzersiz Kullanıcı Sayısı : {df_model["user_id"].nunique():,}')
print(f'• Benzersiz Oyun Sayısı       : {df_model["game_id"].nunique():,}')
print(f'• Pozitif Yorum (5 Puan)      : {(df_model["rating"] == 5).sum():,}')
print(f'• Negatif Yorum (1 Puan)      : {(df_model["rating"] == 1).sum():,}')

# RAM temizliği
del df
import gc
gc.collect()


In [ ]:
# 4. scikit-surprise Veri Formatına Dönüştürme
# Rating skalası [1, 5] aralığında tanımlanır
reader = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(df_model[['user_id', 'game_id', 'rating']], reader)

print('✓ Surprise veri seti başarıyla hazırlandı.')


In [ ]:
# 5. Algoritmaların Tanımlanması ve 3-Katmanlı Çapraz Doğrulama (cv=3)
algorithms = {
    'NormalPredictor (Baseline)': NormalPredictor(),
    'KNNBasic (Cosine)': KNNBasic(k=20, sim_options={'name': 'cosine', 'user_based': False}, verbose=False),
    'NMF': NMF(n_factors=15, n_epochs=15, random_state=42),
    'SVD': SVD(n_factors=20, n_epochs=15, random_state=42)
}

cv_results = []

for name, algo in algorithms.items():
    print(f'Model eğitiliyor: {name} ...')
    t0 = time.time()
    
    cv_out = cross_validate(algo, surprise_data, measures=['RMSE', 'MAE'], cv=3, verbose=False, n_jobs=-1)
    
    duration = time.time() - t0
    mean_rmse = np.mean(cv_out['test_rmse'])
    std_rmse = np.std(cv_out['test_rmse'])
    mean_mae = np.mean(cv_out['test_mae'])
    std_mae = np.std(cv_out['test_mae'])
    
    cv_results.append({
        'Algoritma': name,
        'Test RMSE': round(mean_rmse, 4),
        'RMSE Std': round(std_rmse, 4),
        'Test MAE': round(mean_mae, 4),
        'MAE Std': round(std_mae, 4),
        'Eğitim Süresi (sn)': round(duration, 2)
    })
    print(f'✓ {name} tamamlandı -> RMSE: {mean_rmse:.4f}, MAE: {mean_mae:.4f} ({duration:.2f} sn)')


In [ ]:
# 6. Sonuçların DataFrame Olarak Gösterilmesi
results_df = pd.DataFrame(cv_results)
results_df = results_df.sort_values(by='Test RMSE', ascending=True).reset_index(drop=True)

# Sonuçları kaydet
os.makedirs('outputs', exist_ok=True)
results_df.to_csv('outputs/sonuclar.csv', index=False, encoding='utf-8-sig')

display(results_df)


In [ ]:
# 7. Sonuçların Görselleştirilmesi — Sütun Grafiği (Bar Chart)
colors = ['#27ae60', '#2980b9', '#f39c12', '#c0392b']

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

algo_names = [name.replace(' ', '
') for name in results_df['Algoritma']]
rmse_vals = results_df['Test RMSE'].tolist()
rmse_errs = results_df['RMSE Std'].tolist()

bars = ax.bar(
    algo_names,
    rmse_vals,
    yerr=rmse_errs,
    capsize=7,
    color=colors[:len(algo_names)],
    edgecolor='#2c3e50',
    linewidth=1.2,
    width=0.55,
    alpha=0.9
)

# Sütun tepe değerleri
for bar, val in zip(bars, rmse_vals):
    height = bar.get_height()
    ax.annotate(
        f'{val:.4f}',
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 6),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=12,
        fontweight='bold',
        color='#2c3e50'
    )

ax.set_title('SteamMatch: Algoritmaların RMSE Karşılaştırması
(3-Fold Cross-Validation | 1.000.000 Satır Steam Verisi)', 
             fontsize=13, fontweight='bold', pad=15, color='#2c3e50')
ax.set_xlabel('Öneri Algoritması', fontsize=11, fontweight='semibold', labelpad=10)
ax.set_ylabel('RMSE (Root Mean Squared Error)', fontsize=11, labelpad=10)
ax.set_ylim(0, max(rmse_vals) * 1.25)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('outputs/rmse_karsilastirma.png', dpi=300)
plt.show()


In [ ]:
# 8. En Başarılı Model (SVD) ile Örnek Kullanıcıya Top-5 Oyun Önerisi
full_train = surprise_data.build_full_trainset()
svd_model = SVD(n_factors=20, n_epochs=15, random_state=42)
svd_model.fit(full_train)

# Rastgele bir kullanıcı seç
sample_user_id = df_model['user_id'].iloc[0]
user_played = set(df_model[df_model['user_id'] == sample_user_id]['game_id'])
all_games = df_model['game_id'].unique()
unplayed = [g for g in all_games if g not in user_played]

preds = [svd_model.predict(sample_user_id, g) for g in unplayed]
preds.sort(key=lambda x: x.est, reverse=True)

print(f'👤 Kullanıcı ID: {sample_user_id}')
print(f'🎯 Kullanıcıya Özel En Yüksek Puanlı Top 5 Oyun Önerisi:')
for idx, p in enumerate(preds[:5], start=1):
    print(f'   {idx}. Oyun (App ID): {p.iid:<10} -> Tahmin Edilen Puan: {p.est:.2f} / 5.00')


## 9. Sonuç ve Değerlendirme
- **SVD**, en düşük RMSE (1.2702) ve MAE (0.7772) değerlerini elde ederek kullanıcı ve oyunlar arasındaki gizli özellikleri en iyi yakalayan algoritma olmuştur.
- **NMF** ve **KNNBasic (Cosine)**, rastgele tahmin yapan **NormalPredictor** referans modeline kıyasla hatayı belirgin biçimde azaltmıştır.
- RAM optimizasyonu sayesinde 21+ milyon satırlık devasa veri kümesi başarıyla filtrelenmiş ve 1.000.000 satırlık temiz matris üzerinde saniyeler içinde cross-validation tamamlanmıştır.
